<a href="https://colab.research.google.com/github/minasaeday14/INFO-5731-spring-2026/blob/main/Saeday_Mina_INFO5731_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

TARGET_REVIEWS = 1000
DELAY_SECONDS = 3.2
MOVIE_IDS = ["tt15239678", "tt6263850", "tt22022452"]
OUTPUT_CSV = "q1_imdb_reviews_raw.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

def get_html(url, session):
    response = session.get(url, headers=HEADERS, timeout=30)
    text = response.text

    challenge_markers = [
        "Just a moment",
        "Enable JavaScript and cookies to continue",
        "x-amzn-waf-action"
    ]

    if response.status_code >= 400:
        raise RuntimeError(f"Request failed: {response.status_code} for {url}")

    lower_text = text.lower()
    if response.status_code == 202 and not text.strip():
        raise RuntimeError("IMDb challenge page detected (empty 202 response).")

    for marker in challenge_markers:
        if marker.lower() in lower_text:
            raise RuntimeError("IMDb challenge page detected. Try again later or from a normal browser session/network.")

    time.sleep(DELAY_SECONDS)
    return text

def parse_reviews(html, movie_id):
    import json
    from html import unescape

    soup = BeautifulSoup(html, "html.parser")
    next_data_tag = soup.find("script", id="__NEXT_DATA__")

    rows = []
    end_cursor = None
    has_next_page = False

    if next_data_tag and next_data_tag.string:
        data = json.loads(next_data_tag.string)
        content = data.get("props", {}).get("pageProps", {}).get("contentData", {})

        for item in content.get("reviews", []):
            review = item.get("review", {})
            review_id = review.get("reviewId")
            review_text = unescape(review.get("reviewText", ""))
            review_text = BeautifulSoup(review_text, "html.parser").get_text(" ", strip=True)

            if review_id and review_text:
                rows.append({
                    "movie_id": movie_id,
                    "review_url": f"https://www.imdb.com/title/{movie_id}/review/{review_id}/",
                    "review_text": review_text
                })

        page_info = content.get("pageInfo", {})
        end_cursor = page_info.get("endCursor")
        has_next_page = page_info.get("hasNextPage", False)

    return rows, end_cursor, has_next_page


def fetch_reviews_page_by_cursor(movie_id, cursor, session):
    import json

    variables = {
        "after": cursor,
        "const": movie_id,
        "filter": {},
        "first": 25,
        "isInProfileImageWeblab": False,
        "locale": "en-US",
        "sort": {"by": "HELPFULNESS_SCORE", "order": "DESC"}
    }

    extensions = {
        "persistedQuery": {
            "sha256Hash": "fb58a77d474033025bf28e1fe68f9b998111d3df58e08cd8405bd9265b1a9aff",
            "version": 1
        }
    }

    params = {
        "operationName": "TitleReviewsRefine",
        "variables": json.dumps(variables, separators=(",", ":")),
        "extensions": json.dumps(extensions, separators=(",", ":"))
    }

    headers = dict(HEADERS)
    headers["content-type"] = "application/json"

    response = session.get("https://caching.graphql.imdb.com/", params=params, headers=headers, timeout=30)
    if response.status_code >= 400:
        raise RuntimeError(f"GraphQL request failed: {response.status_code}")

    payload = response.json()
    reviews_payload = payload.get("data", {}).get("title", {}).get("reviews", {})

    rows = []
    for edge in reviews_payload.get("edges", []):
        node = edge.get("node", {})
        review_id = node.get("id")
        plaid_html = node.get("text", {}).get("originalText", {}).get("plaidHtml", "")
        review_text = BeautifulSoup(plaid_html, "html.parser").get_text(" ", strip=True)

        if review_id and review_text:
            rows.append({
                "movie_id": movie_id,
                "review_url": f"https://www.imdb.com/title/{movie_id}/review/{review_id}/",
                "review_text": review_text
            })

    page_info = reviews_payload.get("pageInfo", {})
    end_cursor = page_info.get("endCursor")
    has_next_page = page_info.get("hasNextPage", False)

    time.sleep(DELAY_SECONDS)
    return rows, end_cursor, has_next_page

all_rows = []

with requests.Session() as session:
    for movie_id in MOVIE_IDS:
        print(f"Collecting reviews for {movie_id}...")

        first_page_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
        first_html = get_html(first_page_url, session)
        page_rows, cursor, has_next_page = parse_reviews(first_html, movie_id)
        all_rows.extend(page_rows)
        print(f"  Total collected so far: {len(all_rows)}")

        while len(all_rows) < TARGET_REVIEWS and has_next_page and cursor:
            page_rows, cursor, has_next_page = fetch_reviews_page_by_cursor(movie_id, cursor, session)
            all_rows.extend(page_rows)
            print(f"  Total collected so far: {len(all_rows)}")

        if len(all_rows) >= TARGET_REVIEWS:
            break

df = pd.DataFrame(all_rows)
if not df.empty:
    df = df.drop_duplicates(subset=["review_url"])
    df = df[df["review_text"].str.strip() != ""]

df = df.head(TARGET_REVIEWS)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")

#%%
final_row_count = len(df)
empty_text_count = int((df["review_text"].str.strip() == "").sum()) if not df.empty else 0
duplicate_url_count = int(df["review_url"].duplicated().sum()) if not df.empty else 0

print(f"Final row count: {final_row_count}")
print(f"Empty review_text count: {empty_text_count}")
print(f"Duplicate review_url count: {duplicate_url_count}")

df.head()


  Total collected so far: 25
  Total collected so far: 50
  Total collected so far: 75
  Total collected so far: 100
  Total collected so far: 125
  Total collected so far: 150
  Total collected so far: 175
  Total collected so far: 200
  Total collected so far: 225
  Total collected so far: 250
  Total collected so far: 275
  Total collected so far: 300
  Total collected so far: 325
  Total collected so far: 350
  Total collected so far: 375
  Total collected so far: 400
  Total collected so far: 425
  Total collected so far: 450
  Total collected so far: 475
  Total collected so far: 500
  Total collected so far: 525
  Total collected so far: 550
  Total collected so far: 575
  Total collected so far: 600
  Total collected so far: 625
  Total collected so far: 650
  Total collected so far: 675
  Total collected so far: 700
  Total collected so far: 725
  Total collected so far: 750
  Total collected so far: 775
  Total collected so far: 800
  Total collected so far: 825
  Total colle

,movie_id,review_url,review_text
0,tt15239678,https://www.imdb.com/title/tt15239678/review/r...,I'm going to write this as a review for both D...
1,tt15239678,https://www.imdb.com/title/tt15239678/review/r...,This is what Hollywood needs. A great story wi...
2,tt15239678,https://www.imdb.com/title/tt15239678/review/r...,I have to start by saying that I absolutely lo...
3,tt15239678,https://www.imdb.com/title/tt15239678/review/r...,Had the pleasure to watch this film in an earl...
4,tt15239678,https://www.imdb.com/title/tt15239678/review/r...,Phenomenal stuff. I'll probably calm down tomo...


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
import pandas as pd
import re
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

CSV_PATH = "q1_imdb_reviews_raw.csv"
df = pd.read_csv(CSV_PATH)

if "review_text" not in df.columns:
    raise ValueError("Missing required column: review_text")

print(f"Loaded rows: {len(df)}")
print(f"Columns: {list(df.columns)}")



Loaded rows: 1000
Columns: ['movie_id', 'review_url', 'review_text']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

stopword_set = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def safe_text(text):
    if pd.isna(text):
        return ""
    return str(text)

def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()

def avg_token_count(series):
    return series.fillna("").apply(lambda x: len(str(x).split())).mean()

def compare_step_summary(df, previous_col, new_col):
    before_avg = avg_token_count(df[previous_col])
    after_avg = avg_token_count(df[new_col])
    print(f"Average token count before ({previous_col}): {before_avg:.2f}")
    print(f"Average token count after  ({new_col}): {after_avg:.2f}")

def sample_preview(df, col_a, col_b, n=5, width=120):
    out = df[[col_a, col_b]].head(n).copy()
    out[col_a] = out[col_a].fillna("").apply(lambda x: x[:width])
    out[col_b] = out[col_b].fillna("").apply(lambda x: x[:width])
    return out

def remove_noise(text):
    text = safe_text(text)
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)
    return normalize_spaces(text)

def remove_numbers(text):
    text = safe_text(text)
    text = re.sub(r"\d+", " ", text)
    return normalize_spaces(text)

def remove_stopwords(text, stopword_list):
    text = safe_text(text)
    tokens = text.split()
    filtered = [tok for tok in tokens if tok.lower() not in stopword_list]
    return " ".join(filtered)

def to_lowercase(text):
    return safe_text(text).lower()

def apply_stemming(text, stemmer_obj):
    tokens = safe_text(text).split()
    stemmed = [stemmer_obj.stem(tok) for tok in tokens]
    return " ".join(stemmed)

def apply_lemmatization(text, lemmatizer_obj):
    tokens = safe_text(text).split()
    lemmatized = [lemmatizer_obj.lemmatize(tok) for tok in tokens]
    return " ".join(lemmatized)

In [ ]:
# Step 1: Remove noise (special characters and punctuations)
df["text_step1_no_noise"] = df["review_text"].apply(remove_noise)
compare_step_summary(df, "review_text", "text_step1_no_noise")
print("\nSample preview (review_text -> text_step1_no_noise):")
print(sample_preview(df, "review_text", "text_step1_no_noise").to_string(index=False))

Average token count before (review_text): 219.28
Average token count after  (text_step1_no_noise): 224.88

Sample preview (review_text -> text_step1_no_noise):
                                                                                                             review_text                                                                                                      text_step1_no_noise
I'm going to write this as a review for both Dune movies, so I'll include my thoughts about Dune part one throughout. Fo I m going to write this as a review for both Dune movies so I ll include my thoughts about Dune part one throughout For 
This is what Hollywood needs. A great story with a great director/producer. After that the best thing a studio can do is This is what Hollywood needs A great story with a great director producer After that the best thing a studio can do is g
I have to start by saying that I absolutely loved the first movie. I wasn't familiar with the story and I went wit

In [ ]:
# Step 2: Remove numbers
df["text_step2_no_numbers"] = df["text_step1_no_noise"].apply(remove_numbers)
compare_step_summary(df, "text_step1_no_noise", "text_step2_no_numbers")
digit_rows_after_step2 = int(df["text_step2_no_numbers"].str.contains(r"\d", regex=True).sum())
print(f"Rows that still contain digits after Step 2: {digit_rows_after_step2}")
print("\nSample preview (text_step1_no_noise -> text_step2_no_numbers):")
print(sample_preview(df, "text_step1_no_noise", "text_step2_no_numbers").to_string(index=False))

Average token count before (text_step1_no_noise): 224.88
Average token count after  (text_step2_no_numbers): 222.83
Rows that still contain digits after Step 2: 0

Sample preview (text_step1_no_noise -> text_step2_no_numbers):
                                                                                                     text_step1_no_noise                                                                                                    text_step2_no_numbers
I m going to write this as a review for both Dune movies so I ll include my thoughts about Dune part one throughout For  I m going to write this as a review for both Dune movies so I ll include my thoughts about Dune part one throughout For 
This is what Hollywood needs A great story with a great director producer After that the best thing a studio can do is g This is what Hollywood needs A great story with a great director producer After that the best thing a studio can do is g
I have to start by saying that I absolutely lov

In [ ]:
# Step 3: Remove stopwords
df["text_step3_no_stopwords"] = df["text_step2_no_numbers"].apply(lambda x: remove_stopwords(x, stopword_set))
compare_step_summary(df, "text_step2_no_numbers", "text_step3_no_stopwords")
print("\nSample preview (text_step2_no_numbers -> text_step3_no_stopwords):")
print(sample_preview(df, "text_step2_no_numbers", "text_step3_no_stopwords").to_string(index=False))

Average token count before (text_step2_no_numbers): 222.83
Average token count after  (text_step3_no_stopwords): 113.58

Sample preview (text_step2_no_numbers -> text_step3_no_stopwords):
                                                                                                   text_step2_no_numbers                                                                                                  text_step3_no_stopwords
I m going to write this as a review for both Dune movies so I ll include my thoughts about Dune part one throughout For  going write review Dune movies include thoughts Dune part one throughout difficult part rating movies trying judge stand
This is what Hollywood needs A great story with a great director producer After that the best thing a studio can do is g Hollywood needs great story great director producer best thing studio get hell way let artists create art Dune Part crea
I have to start by saying that I absolutely loved the first movie I wasn t familiar wi

In [ ]:
# Step 4: Lowercase all text
df["text_step4_lowercase"] = df["text_step3_no_stopwords"].apply(to_lowercase)
compare_step_summary(df, "text_step3_no_stopwords", "text_step4_lowercase")
not_lowercase_count = int((df["text_step4_lowercase"] != df["text_step4_lowercase"].str.lower()).sum())
print(f"Rows not fully lowercase after Step 4: {not_lowercase_count}")
print("\nSample preview (text_step3_no_stopwords -> text_step4_lowercase):")
print(sample_preview(df, "text_step3_no_stopwords", "text_step4_lowercase").to_string(index=False))

Average token count before (text_step3_no_stopwords): 113.58
Average token count after  (text_step4_lowercase): 113.58
Rows not fully lowercase after Step 4: 0

Sample preview (text_step3_no_stopwords -> text_step4_lowercase):
                                                                                                 text_step3_no_stopwords                                                                                                     text_step4_lowercase
going write review Dune movies include thoughts Dune part one throughout difficult part rating movies trying judge stand going write review dune movies include thoughts dune part one throughout difficult part rating movies trying judge stand
Hollywood needs great story great director producer best thing studio get hell way let artists create art Dune Part crea hollywood needs great story great director producer best thing studio get hell way let artists create art dune part crea
start saying absolutely loved first movie famil

In [ ]:
# Step 5: Stemming
df["text_step5_stemmed"] = df["text_step4_lowercase"].apply(lambda x: apply_stemming(x, stemmer))
compare_step_summary(df, "text_step4_lowercase", "text_step5_stemmed")
print("\nSample preview (text_step4_lowercase -> text_step5_stemmed):")
print(sample_preview(df, "text_step4_lowercase", "text_step5_stemmed").to_string(index=False))

Average token count before (text_step4_lowercase): 113.58
Average token count after  (text_step5_stemmed): 113.58

Sample preview (text_step4_lowercase -> text_step5_stemmed):
                                                                                                    text_step4_lowercase                                                                                                       text_step5_stemmed
going write review dune movies include thoughts dune part one throughout difficult part rating movies trying judge stand go write review dune movi includ thought dune part one throughout difficult part rate movi tri judg stand without compar
hollywood needs great story great director producer best thing studio get hell way let artists create art dune part crea hollywood need great stori great director produc best thing studio get hell way let artist creat art dune part creativ b
start saying absolutely loved first movie familiar story went zero expectations completely stunned

In [ ]:
# Step 6: Lemmatization
df["clean_text"] = df["text_step5_stemmed"].apply(lambda x: apply_lemmatization(x, lemmatizer))
compare_step_summary(df, "text_step5_stemmed", "clean_text")
print("\nSample preview (text_step5_stemmed -> clean_text):")
print(sample_preview(df, "text_step5_stemmed", "clean_text").to_string(index=False))

Average token count before (text_step5_stemmed): 113.58
Average token count after  (clean_text): 113.58

Sample preview (text_step5_stemmed -> clean_text):
                                                                                                      text_step5_stemmed                                                                                                               clean_text
go write review dune movi includ thought dune part one throughout difficult part rate movi tri judg stand without compar go write review dune movi includ thought dune part one throughout difficult part rate movi tri judg stand without compar
hollywood need great stori great director produc best thing studio get hell way let artist creat art dune part creativ b hollywood need great stori great director produc best thing studio get hell way let artist creat art dune part creativ b
start say absolut love first movi familiar stori went zero expect complet stun master artwork dune part music sound vi

In [ ]:
# Save back to the same CSV file
df.to_csv(CSV_PATH, index=False)

# Final validation checks
check_df = pd.read_csv(CSV_PATH)
digits_in_step2 = int(check_df['text_step2_no_numbers'].fillna('').str.contains(r'\d', regex=True).sum())

print(f"Saved file: {CSV_PATH}")
print(f"Final row count: {len(check_df)}")
print(f"Duplicate review_url count: {int(check_df['review_url'].duplicated().sum())}")
print(f"Rows with empty clean_text: {int((check_df['clean_text'].fillna('').str.strip() == '').sum())}")
print(f"Rows with digits in text_step2_no_numbers: {digits_in_step2}")
print("\nFinal columns:")
print(list(check_df.columns))

Saved file: q1_imdb_reviews_raw.csv
Final row count: 1000
Duplicate review_url count: 0
Rows with empty clean_text: 0
Rows with digits in text_step2_no_numbers: 0

Final columns:
['movie_id', 'review_url', 'review_text', 'text_step1_no_noise', 'text_step2_no_numbers', 'text_step3_no_stopwords', 'text_step4_lowercase', 'text_step5_stemmed', 'clean_text']


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
import sys
import subprocess

try:
    import stanza
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "stanza"])
    import stanza

import os
import re
from collections import Counter

import pandas as pd

In [ ]:
# Download English models once
stanza.download(
    "en",
    processors="tokenize,pos,lemma,depparse,constituency,ner",
    verbose=False,
)

nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos,lemma,depparse,constituency,ner",
    use_gpu=False,
    verbose=False,
)

print("Stanza pipeline ready.")

# Lightweight POS-only pipeline for supplemental clean_text comparison
nlp_pos = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos",
    use_gpu=False,
    verbose=False,
)

Stanza pipeline ready.


In [ ]:
INPUT_CSV = "q1_imdb_reviews_raw.csv"
CONSTITUENCY_TXT = "q3_constituency_trees.txt"
DEPENDENCY_TXT = "q3_dependency_trees.txt"
POS_SUMMARY_CSV = "q3_pos_summary.csv"
NER_SUMMARY_CSV = "q3_ner_summary.csv"
PARSE_ERRORS_CSV = "q3_parse_errors.csv"

df = pd.read_csv(INPUT_CSV)
required_cols = ["review_url", "review_text", "clean_text"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.fillna("")
for c in required_cols:
    df[c] = df[c].astype(str)

df = df[df["review_text"].str.strip() != ""].reset_index(drop=True)

print(f"Rows loaded for Q3: {len(df)}")
print("Columns:", list(df.columns))

Rows loaded for Q3: 1000
Columns: ['movie_id', 'review_url', 'review_text', 'text_step1_no_noise', 'text_step2_no_numbers', 'text_step3_no_stopwords', 'text_step4_lowercase', 'text_step5_stemmed', 'clean_text']


In [ ]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

upos_counter_review = Counter()
upos_counter_clean = Counter()

entity_category_counter = Counter({
    "person": 0,
    "organization": 0,
    "location": 0,
    "product": 0,
    "date": 0,
})

parse_errors = []

reviews_processed = 0
sentences_processed = 0
constituency_records_written = 0
dependency_records_written = 0

constituency_preview = []
dependency_preview = []

with open(CONSTITUENCY_TXT, "w", encoding="utf-8") as const_out, open(DEPENDENCY_TXT, "w", encoding="utf-8") as dep_out:
    for row_idx, row in df.iterrows():
        review_url = row["review_url"]
        review_text = normalize_text(row["review_text"])

        if not review_text:
            continue

        try:
            doc = nlp(review_text)
            reviews_processed += 1
        except Exception as e:
            parse_errors.append({
                "source": "review_text",
                "row_index": row_idx,
                "review_url": review_url,
                "sentence_index": None,
                "stage": "document_parse",
                "error_message": str(e),
            })
            continue

        # NER counts
        try:
            for ent in doc.ents:
                if ent.type == "PERSON":
                    entity_category_counter["person"] += 1
                elif ent.type == "ORG":
                    entity_category_counter["organization"] += 1
                elif ent.type in {"GPE", "LOC", "FAC"}:
                    entity_category_counter["location"] += 1
                elif ent.type == "PRODUCT":
                    entity_category_counter["product"] += 1
                elif ent.type == "DATE":
                    entity_category_counter["date"] += 1
        except Exception as e:
            parse_errors.append({
                "source": "review_text",
                "row_index": row_idx,
                "review_url": review_url,
                "sentence_index": None,
                "stage": "ner_extract",
                "error_message": str(e),
            })

        # Sentence-level POS + parsing trees
        for sent_idx, sent in enumerate(doc.sentences, start=1):
            sent_text = sent.text.strip()
            if not sent_text:
                continue

            sentences_processed += 1

            for word in sent.words:
                if word.upos:
                    upos_counter_review[word.upos] += 1

            try:
                constituency_tree = str(sent.constituency) if sent.constituency is not None else ""
            except Exception as e:
                constituency_tree = ""
                parse_errors.append({
                    "source": "review_text",
                    "row_index": row_idx,
                    "review_url": review_url,
                    "sentence_index": sent_idx,
                    "stage": "constituency_extract",
                    "error_message": str(e),
                })

            dep_edges = []
            try:
                for w in sent.words:
                    head_text = "ROOT" if w.head == 0 else sent.words[w.head - 1].text
                    dep_edges.append(f"{w.text} <-{w.deprel}- {head_text}")
            except Exception as e:
                parse_errors.append({
                    "source": "review_text",
                    "row_index": row_idx,
                    "review_url": review_url,
                    "sentence_index": sent_idx,
                    "stage": "dependency_extract",
                    "error_message": str(e),
                })

            const_out.write(f"Review URL: {review_url}\n")
            const_out.write(f"Sentence #: {sent_idx}\n")
            const_out.write(f"Sentence: {sent_text}\n")
            const_out.write("Constituency Tree:\n")
            const_out.write(f"{constituency_tree}\n")
            const_out.write("-" * 80 + "\n")
            constituency_records_written += 1

            dep_text = " | ".join(dep_edges)
            dep_out.write(f"Review URL: {review_url}\n")
            dep_out.write(f"Sentence #: {sent_idx}\n")
            dep_out.write(f"Sentence: {sent_text}\n")
            dep_out.write("Dependency Relations:\n")
            dep_out.write(f"{dep_text}\n")
            dep_out.write("-" * 80 + "\n")
            dependency_records_written += 1

            if len(constituency_preview) < 3:
                constituency_preview.append({
                    "review_url": review_url,
                    "sentence_index": sent_idx,
                    "sentence_text": sent_text,
                    "constituency_tree": constituency_tree,
                })
            if len(dependency_preview) < 3:
                dependency_preview.append({
                    "review_url": review_url,
                    "sentence_index": sent_idx,
                    "sentence_text": sent_text,
                    "dependency_edges": dep_text,
                })

print(f"Processed reviews: {reviews_processed}")
print(f"Processed sentences: {sentences_processed}")
print(f"Constituency records written: {constituency_records_written}")
print(f"Dependency records written: {dependency_records_written}")
print(f"Current parse errors: {len(parse_errors)}")

KeyboardInterrupt: 

In [ ]:
for row_idx, row in df.iterrows():
    clean_text = normalize_text(row["clean_text"])
    if not clean_text:
        continue

    try:
        doc_clean = nlp_pos(clean_text)
    except Exception as e:
        parse_errors.append({
            "source": "clean_text",
            "row_index": row_idx,
            "review_url": row["review_url"],
            "sentence_index": None,
            "stage": "document_parse",
            "error_message": str(e),
        })
        continue

    for sent in doc_clean.sentences:
        for word in sent.words:
            if word.upos:
                upos_counter_clean[word.upos] += 1

print("Supplemental clean_text POS completed.")

In [ ]:
def pos_totals(counter_obj: Counter):
    return {
        "noun": counter_obj.get("NOUN", 0) + counter_obj.get("PROPN", 0),
        "verb": counter_obj.get("VERB", 0) + counter_obj.get("AUX", 0),
        "adjective": counter_obj.get("ADJ", 0),
        "adverb": counter_obj.get("ADV", 0),
    }

review_pos = pos_totals(upos_counter_review)
clean_pos = pos_totals(upos_counter_clean)

pos_summary_df = pd.DataFrame([
    {"category": "N", "count": review_pos["noun"]},
    {"category": "V", "count": review_pos["verb"]},
    {"category": "Adj", "count": review_pos["adjective"]},
    {"category": "Adv", "count": review_pos["adverb"]},
])
pos_summary_df.to_csv(POS_SUMMARY_CSV, index=False)

print("POS totals from review_text")
print(pos_summary_df)

pos_compare_df = pd.DataFrame([
    {
        "source": "review_text",
        "N": review_pos["noun"],
        "V": review_pos["verb"],
        "Adj": review_pos["adjective"],
        "Adv": review_pos["adverb"],
    },
    {
        "source": "clean_text",
        "N": clean_pos["noun"],
        "V": clean_pos["verb"],
        "Adj": clean_pos["adjective"],
        "Adv": clean_pos["adverb"],
    },
])

print("\nPOS comparison (dual-track)")
print(pos_compare_df)

In [ ]:
print("Constituency examples (first 3)\n")
for rec in constituency_preview:
    print(f"Review URL: {rec['review_url']}")
    print(f"Sentence #: {rec['sentence_index']}")
    print(f"Sentence: {rec['sentence_text']}")
    print(f"Tree: {rec['constituency_tree']}")
    print("-" * 60)

print("\nDependency examples (first 3)\n")
for rec in dependency_preview:
    print(f"Review URL: {rec['review_url']}")
    print(f"Sentence #: {rec['sentence_index']}")
    print(f"Sentence: {rec['sentence_text']}")
    print(f"Dependencies: {rec['dependency_edges']}")
    print("-" * 60)

In [ ]:
if constituency_preview and dependency_preview:
    sample_sentence = constituency_preview[0]["sentence_text"]

    print("Example sentence:")
    print(sample_sentence)
    print("\nConstituency understanding:")
    print(
        "The constituency tree groups words into nested phrases (NP, VP, PP, etc.), "
        "showing phrase structure hierarchy."
    )
    print("\nDependency understanding:")
    print(
        "The dependency tree links each word to a head word with relation labels "
        "(for example nsubj, obj, amod), showing grammatical dependencies."
    )
else:
    print("No parsed sentence available for explanation.")

In [ ]:
ner_summary_df = pd.DataFrame([
    {"entity_category": "person", "count": entity_category_counter["person"]},
    {"entity_category": "organization", "count": entity_category_counter["organization"]},
    {"entity_category": "location", "count": entity_category_counter["location"]},
    {"entity_category": "product", "count": entity_category_counter["product"]},
    {"entity_category": "date", "count": entity_category_counter["date"]},
])
ner_summary_df.to_csv(NER_SUMMARY_CSV, index=False)

print("NER category counts")
print(ner_summary_df)

In [ ]:
parse_errors_df = pd.DataFrame(parse_errors)
if parse_errors_df.empty:
    parse_errors_df = pd.DataFrame(columns=[
        "source", "row_index", "review_url", "sentence_index", "stage", "error_message"
    ])
parse_errors_df.to_csv(PARSE_ERRORS_CSV, index=False)

print(f"Saved parse errors: {PARSE_ERRORS_CSV}")
print(f"Error rows: {len(parse_errors_df)}")

In [ ]:
output_files = [
    CONSTITUENCY_TXT,
    DEPENDENCY_TXT,
    POS_SUMMARY_CSV,
    NER_SUMMARY_CSV,
    PARSE_ERRORS_CSV,
]

print("Final QA")
print(f"Reviews processed: {reviews_processed}")
print(f"Sentences processed: {sentences_processed}")
print(f"Constituency records written: {constituency_records_written}")
print(f"Dependency records written: {dependency_records_written}")
print(f"Parse errors: {len(parse_errors_df)}")

for fp in output_files:
    print(f"Exists: {os.path.exists(fp)} -> {fp}")

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [ ]:
import time
import re
from typing import Dict, List, Tuple

import requests
from bs4 import BeautifulSoup
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Imports and NLTK setup complete.")

Imports and NLTK setup complete.


In [ ]:
TARGET_PRODUCTS = 1000
REQUEST_DELAY_SECONDS = 1.1
START_PAGE = 1
MAX_PAGE = 500

BASE_URL = "https://github.com/marketplace?type=actions&page={page}"

RAW_CSV = "q4_marketplace_raw.csv"
CLEAN_CSV = "q4_marketplace_clean.csv"
QUALITY_CSV = "q4_marketplace_quality_report.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}

print("Constants are set.")

Constants are set.


In [ ]:
def fetch_page(page_number: int, retries: int = 2, timeout: int = 30) -> Tuple[str, int, str]:
    url = BASE_URL.format(page=page_number)
    last_error = ""

    for attempt in range(retries + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=timeout)
            if response.status_code == 200:
                return response.text, response.status_code, ""
            last_error = f"HTTP {response.status_code}"
        except Exception as e:
            last_error = str(e)

        if attempt < retries:
            time.sleep(1.5)

    return "", -1, last_error


def parse_marketplace_items(html: str, page_number: int) -> List[Dict[str, str]]:
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select('div[data-testid="marketplace-item"]')

    items = []
    for card in cards:
        link = card.select_one('h3 a[href*="/marketplace/actions/"]')
        desc_tag = card.select_one("p")

        if link is None:
            continue

        href = link.get("href", "").strip()
        if not href:
            continue

        if href.startswith("http"):
            full_url = href
        else:
            full_url = "https://github.com" + href

        name = link.get_text(" ", strip=True)
        description = desc_tag.get_text(" ", strip=True) if desc_tag else ""

        items.append({
            "product_name": name,
            "description": description,
            "url": full_url,
            "page_number": page_number,
        })

    return items

In [ ]:
rows: List[Dict[str, str]] = []
seen_urls = set()
page_errors = []
page_card_counts: Dict[int, int] = {}
pages_requested = 0

for page in range(START_PAGE, MAX_PAGE + 1):
    html, status, error_message = fetch_page(page)
    pages_requested += 1

    if not html:
        page_errors.append({"page": page, "status": status, "error": error_message})
        print(f"Page {page}: request failed -> {error_message}")
        time.sleep(REQUEST_DELAY_SECONDS)
        continue

    items = parse_marketplace_items(html, page)
    page_card_counts[page] = len(items)

    if len(items) == 0:
        print(f"Page {page}: no items found. Stopping pagination.")
        break

    for item in items:
        rows.append(item)
        seen_urls.add(item["url"])

    if page % 5 == 0 or page <= 3:
        print(f"Page {page}: items={len(items)}, unique_urls={len(seen_urls)}")

    if len(seen_urls) >= TARGET_PRODUCTS:
        print(f"Reached target unique URL count at page {page}.")
        break

    time.sleep(REQUEST_DELAY_SECONDS)

print("\nScraping finished.")
print(f"Pages requested: {pages_requested}")
print(f"Rows scraped (before dedupe): {len(rows)}")
print(f"Unique URLs collected: {len(seen_urls)}")
print(f"Page errors: {len(page_errors)}")

Page 1: items=20, unique_urls=20
Page 2: items=20, unique_urls=40
Page 3: items=20, unique_urls=60
Page 5: items=20, unique_urls=100
Page 10: items=20, unique_urls=200
Page 15: items=20, unique_urls=300
Page 20: items=20, unique_urls=400
Page 25: items=20, unique_urls=498
Page 30: items=20, unique_urls=597
Page 35: items=20, unique_urls=696
Page 40: items=20, unique_urls=795
Page 45: items=20, unique_urls=895
Page 50: items=20, unique_urls=995
Reached target unique URL count at page 51.

Scraping finished.
Pages requested: 51
Rows scraped (before dedupe): 1020
Unique URLs collected: 1015
Page errors: 0


In [ ]:
raw_df = pd.DataFrame(rows)

if raw_df.empty:
    raise ValueError("No data was scraped. Check network/selectors and rerun.")

raw_before_dedupe = len(raw_df)
raw_df = raw_df.drop_duplicates(subset=["url"], keep="first").reset_index(drop=True)
raw_after_dedupe = len(raw_df)

if raw_after_dedupe >= TARGET_PRODUCTS:
    raw_df = raw_df.head(TARGET_PRODUCTS).copy()

raw_df.to_csv(RAW_CSV, index=False)

print(f"Rows before dedupe: {raw_before_dedupe}")
print(f"Rows after dedupe: {raw_after_dedupe}")
print(f"Final raw rows saved: {len(raw_df)}")
print(f"Saved raw CSV: {RAW_CSV}")

Rows before dedupe: 1020
Rows after dedupe: 1015
Final raw rows saved: 1000
Saved raw CSV: q4_marketplace_raw.csv


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text: str) -> Tuple[str, str]:
    if text is None:
        text = ""
    text = str(text)

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()

    tokens = []
    for tok in text.split():
        if tok in stop_words:
            continue
        tokens.append(lemmatizer.lemmatize(tok))

    clean_description = " ".join(tokens)
    token_string = " ".join(tokens)
    return clean_description, token_string


clean_df = raw_df.copy()
clean_results = clean_df["description"].apply(preprocess_text)
clean_df[["clean_description", "tokens"]] = pd.DataFrame(clean_results.tolist(), index=clean_df.index)

clean_df.to_csv(CLEAN_CSV, index=False)

print(f"Saved clean CSV: {CLEAN_CSV}")
print(f"Clean rows: {len(clean_df)}")

Saved clean CSV: q4_marketplace_clean.csv
Clean rows: 1000


In [ ]:
quality_rows = []

required_cols_raw = ["product_name", "description", "url", "page_number"]
required_cols_clean = required_cols_raw + ["clean_description", "tokens"]

missing_raw_cols = [c for c in required_cols_raw if c not in raw_df.columns]
quality_rows.append({
    "check_name": "required_columns_raw",
    "value": len(missing_raw_cols),
    "status": "PASS" if len(missing_raw_cols) == 0 else "FAIL",
    "notes": "Missing: " + (", ".join(missing_raw_cols) if missing_raw_cols else "none"),
})

missing_clean_cols = [c for c in required_cols_clean if c not in clean_df.columns]
quality_rows.append({
    "check_name": "required_columns_clean",
    "value": len(missing_clean_cols),
    "status": "PASS" if len(missing_clean_cols) == 0 else "FAIL",
    "notes": "Missing: " + (", ".join(missing_clean_cols) if missing_clean_cols else "none"),
})

for col in required_cols_raw:
    missing_count = int(raw_df[col].isna().sum())
    if raw_df[col].dtype == object:
        missing_count += int((raw_df[col].astype(str).str.strip() == "").sum())
    quality_rows.append({
        "check_name": f"missing_{col}",
        "value": missing_count,
        "status": "PASS" if missing_count == 0 else "WARN",
        "notes": "Raw dataset missing/blank count",
    })

dup_raw = int(raw_df.duplicated(subset=["url"]).sum())
dup_clean = int(clean_df.duplicated(subset=["url"]).sum())
quality_rows.append({
    "check_name": "duplicate_url_raw",
    "value": dup_raw,
    "status": "PASS" if dup_raw == 0 else "FAIL",
    "notes": "Duplicate URLs in raw output",
})
quality_rows.append({
    "check_name": "duplicate_url_clean",
    "value": dup_clean,
    "status": "PASS" if dup_clean == 0 else "FAIL",
    "notes": "Duplicate URLs in clean output",
})

empty_desc = int((raw_df["description"].astype(str).str.strip() == "").sum())
quality_rows.append({
    "check_name": "empty_description_raw",
    "value": empty_desc,
    "status": "PASS" if empty_desc == 0 else "WARN",
    "notes": "Blank descriptions in raw output",
})

valid_prefix = "https://github.com/marketplace/actions/"
invalid_url_count = int((~clean_df["url"].astype(str).str.startswith(valid_prefix)).sum())
quality_rows.append({
    "check_name": "invalid_url_format",
    "value": invalid_url_count,
    "status": "PASS" if invalid_url_count == 0 else "WARN",
    "notes": f"URL must start with {valid_prefix}",
})

final_rows = len(clean_df)
quality_rows.append({
    "check_name": "final_row_count_equals_1000",
    "value": final_rows,
    "status": "PASS" if final_rows == TARGET_PRODUCTS else "WARN",
    "notes": "Expected exactly 1000 if marketplace has enough unique items",
})

for p in [1, 2, 3]:
    count = page_card_counts.get(p, -1)
    quality_rows.append({
        "check_name": f"page_{p}_parse_items_count",
        "value": count,
        "status": "PASS" if count > 0 else "WARN",
        "notes": "Expected positive item count for early pages",
    })

quality_rows.append({
    "check_name": "request_failures",
    "value": len(page_errors),
    "status": "PASS" if len(page_errors) == 0 else "WARN",
    "notes": "Failed page requests after retries",
})

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(QUALITY_CSV, index=False)

print(f"Saved quality report: {QUALITY_CSV}")
print("\nQuality report preview:")
print(quality_df.head(12).to_string(index=False))

Saved quality report: q4_marketplace_quality_report.csv

Quality report preview:
                 check_name  value status                                                        notes
       required_columns_raw      0   PASS                                                Missing: none
     required_columns_clean      0   PASS                                                Missing: none
       missing_product_name      0   PASS                              Raw dataset missing/blank count
        missing_description      1   WARN                              Raw dataset missing/blank count
                missing_url      0   PASS                              Raw dataset missing/blank count
        missing_page_number      0   PASS                              Raw dataset missing/blank count
          duplicate_url_raw      0   PASS                                 Duplicate URLs in raw output
        duplicate_url_clean      0   PASS                               Duplicate URLs in clean

In [ ]:
print("Summary")
print(f"Pages requested: {pages_requested}")
print(f"Rows scraped before dedupe: {raw_before_dedupe}")
print(f"Rows after dedupe: {raw_after_dedupe}")
print(f"Final rows in raw/clean CSV: {len(clean_df)}")
print(f"Duplicates removed: {raw_before_dedupe - raw_after_dedupe}")

print("\nMissing-value summary (raw):")
print(raw_df[["product_name", "description", "url", "page_number"]].isna().sum().to_string())

print("\nSample raw rows:")
print(raw_df[["product_name", "description", "url", "page_number"]].head(5).to_string(index=False))

print("\nSample cleaned rows:")
print(clean_df[["description", "clean_description", "tokens"]].head(5).to_string(index=False))

Summary
Pages requested: 51
Rows scraped before dedupe: 1020
Rows after dedupe: 1015
Final rows in raw/clean CSV: 1000
Duplicates removed: 5

Missing-value summary (raw):
product_name    0
description     0
url             0
page_number     0

Sample raw rows:
                product_name                                                                                                description                                                               url  page_number
              TruffleHog OSS                                                     Find and verify leaked credentials in your source code             https://github.com/marketplace/actions/trufflehog-oss            1
               Metrics embed     An infographics generator with 40+ plugins and 300+ options to display stats about your GitHub account              https://github.com/marketplace/actions/metrics-embed            1
yq - portable yaml processor                                        create, read, update, dele

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [ ]:
import sys
import subprocess
import re
import time
from datetime import datetime

import pandas as pd
import nltk

try:
    import tweepy
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tweepy"])
    import tweepy

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Imports complete.")

Imports complete.


In [ ]:
from google.colab import userdata
import tweepy

BEARER_TOKEN = userdata.get("X_BEARER_TOKEN")

TARGET_TWEETS = 500
MAX_RESULTS_PER_CALL = 100
QUERY = "(#machinelearning OR #artificialintelligence) -is:retweet lang:en"

RAW_CSV = "q5_tweets_raw.csv"
CLEAN_CSV = "q5_tweets_clean.csv"
QUALITY_CSV = "q5_tweets_quality_report.csv"

client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)
print("Config set.")

Config set.


In [ ]:
auth_ok = False
preflight_error = ""

try:
    _ = client.search_recent_tweets(
        query=QUERY,
        tweet_fields=["author_id", "created_at", "lang"],
        expansions=["author_id"],
        user_fields=["username"],
        max_results=10,
    )
    auth_ok = True
    print("Preflight success.")
except Exception as e:
    preflight_error = str(e)
    print("Preflight failed:", preflight_error)
    print("Check X_BEARER_TOKEN secret, app access, and billing in https://console.x.com")


Preflight failed: 402 Payment Required
Your enrolled account [2028714052411949056] does not have any credits to fulfill this request.
Check X_BEARER_TOKEN secret, app access, and billing in https://console.x.com


In [ ]:
rows = []
seen_tweet_ids = set()
pages_used = 0

if not auth_ok:
    print("Skipping tweet collection because preflight did not pass.")
else:
    next_token = None

    while len(seen_tweet_ids) < TARGET_TWEETS:
        try:
            response = client.search_recent_tweets(
                query=QUERY,
                tweet_fields=["author_id", "created_at", "lang"],
                expansions=["author_id"],
                user_fields=["username"],
                max_results=MAX_RESULTS_PER_CALL,
                next_token=next_token,
            )
            pages_used += 1
        except Exception as e:
            print("Collection stopped due to API error:", str(e))
            break

        tweets = response.data if response and response.data is not None else []
        if not tweets:
            print("No more tweets returned.")
            break

        users_map = {}
        if response.includes and "users" in response.includes:
            for user in response.includes["users"]:
                users_map[user.id] = user.username

        for tweet in tweets:
            tweet_id = str(tweet.id)
            if tweet_id in seen_tweet_ids:
                continue

            seen_tweet_ids.add(tweet_id)
            username = users_map.get(tweet.author_id, "unknown")
            created_at = ""
            if tweet.created_at is not None:
                created_at = tweet.created_at.isoformat()

            rows.append({
                "tweet_id": tweet_id,
                "username": username,
                "text": tweet.text,
                "created_at": created_at,
                "query": QUERY,
                "collected_at": datetime.utcnow().isoformat(),
            })

            if len(seen_tweet_ids) >= TARGET_TWEETS:
                break

        meta = response.meta if response and response.meta is not None else {}
        next_token = meta.get("next_token")
        if not next_token:
            print("No next_token returned. Pagination finished.")
            break

        time.sleep(1.0)

print("Collection summary")
print("Pages used:", pages_used)
print("Rows collected (before dataframe dedupe):", len(rows))
print("Unique tweet IDs collected:", len(seen_tweet_ids))

Skipping tweet collection because preflight did not pass.
Collection summary
Pages used: 0
Rows collected (before dataframe dedupe): 0
Unique tweet IDs collected: 0


In [ ]:
required_raw_cols = ["tweet_id", "username", "text", "created_at", "query", "collected_at"]

raw_df = pd.DataFrame(rows, columns=required_raw_cols)
raw_before_dedupe = len(raw_df)

if not raw_df.empty:
    raw_df = raw_df.drop_duplicates(subset=["tweet_id"], keep="first").reset_index(drop=True)

raw_after_dedupe = len(raw_df)

if raw_after_dedupe > TARGET_TWEETS:
    raw_df = raw_df.head(TARGET_TWEETS).copy()

raw_df.to_csv(RAW_CSV, index=False)

print("Raw save summary")
print("Rows before dedupe:", raw_before_dedupe)
print("Rows after dedupe:", raw_after_dedupe)
print("Rows saved:", len(raw_df))
print("Saved:", RAW_CSV)

Raw save summary
Rows before dedupe: 0
Rows after dedupe: 0
Rows saved: 0
Saved: q5_tweets_raw.csv


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_tweet_text(text: str):
    text = "" if text is None else str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("#", " ")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for tok in text.split():
        if tok in stop_words:
            continue
        lemma = lemmatizer.lemmatize(tok)
        if lemma:
            tokens.append(lemma)

    clean_text = " ".join(tokens)
    token_string = " ".join(tokens)
    return clean_text, token_string


clean_df = raw_df.copy()

if clean_df.empty:
    clean_df["clean_text"] = pd.Series(dtype=str)
    clean_df["tokens"] = pd.Series(dtype=str)
else:
    cleaned_pairs = clean_df["text"].apply(clean_tweet_text)
    clean_df[["clean_text", "tokens"]] = pd.DataFrame(cleaned_pairs.tolist(), index=clean_df.index)

clean_df.to_csv(CLEAN_CSV, index=False)

print("Cleaning summary")
print("Rows in clean dataset:", len(clean_df))
print("Saved:", CLEAN_CSV)

Cleaning summary
Rows in clean dataset: 0
Saved: q5_tweets_clean.csv


In [ ]:
quality_rows = []

required_raw_cols = ["tweet_id", "username", "text", "created_at", "query", "collected_at"]
required_clean_cols = required_raw_cols + ["clean_text", "tokens"]

missing_raw_cols = [c for c in required_raw_cols if c not in raw_df.columns]
quality_rows.append({
    "check_name": "required_columns_raw",
    "value": len(missing_raw_cols),
    "status": "PASS" if len(missing_raw_cols) == 0 else "FAIL",
    "notes": "Missing: " + (", ".join(missing_raw_cols) if missing_raw_cols else "none"),
})

missing_clean_cols = [c for c in required_clean_cols if c not in clean_df.columns]
quality_rows.append({
    "check_name": "required_columns_clean",
    "value": len(missing_clean_cols),
    "status": "PASS" if len(missing_clean_cols) == 0 else "FAIL",
    "notes": "Missing: " + (", ".join(missing_clean_cols) if missing_clean_cols else "none"),
})

quality_rows.append({
    "check_name": "api_preflight",
    "value": 1 if auth_ok else 0,
    "status": "PASS" if auth_ok else "WARN",
    "notes": "Preflight API test status. " + ("OK" if auth_ok else preflight_error),
})

for col in ["tweet_id", "username", "text"]:
    if col in raw_df.columns:
        blank_count = int(raw_df[col].fillna("").astype(str).str.strip().eq("").sum())
    else:
        blank_count = -1

    quality_rows.append({
        "check_name": f"missing_or_blank_{col}",
        "value": blank_count,
        "status": "PASS" if blank_count == 0 else "WARN",
        "notes": "Raw dataset blank check",
    })

if "clean_text" in clean_df.columns:
    blank_clean = int(clean_df["clean_text"].fillna("").astype(str).str.strip().eq("").sum())
else:
    blank_clean = -1

quality_rows.append({
    "check_name": "missing_or_blank_clean_text",
    "value": blank_clean,
    "status": "PASS" if blank_clean == 0 else "WARN",
    "notes": "Clean dataset blank check",
})

dup_raw = int(raw_df.duplicated(subset=["tweet_id"]).sum()) if "tweet_id" in raw_df.columns else -1
dup_clean = int(clean_df.duplicated(subset=["tweet_id"]).sum()) if "tweet_id" in clean_df.columns else -1

quality_rows.append({
    "check_name": "duplicate_tweet_id_raw",
    "value": dup_raw,
    "status": "PASS" if dup_raw == 0 else "WARN",
    "notes": "Duplicate tweet IDs in raw",
})
quality_rows.append({
    "check_name": "duplicate_tweet_id_clean",
    "value": dup_clean,
    "status": "PASS" if dup_clean == 0 else "WARN",
    "notes": "Duplicate tweet IDs in clean",
})

final_rows = len(clean_df)
quality_rows.append({
    "check_name": "final_row_count",
    "value": final_rows,
    "status": "PASS" if final_rows == TARGET_TWEETS else "WARN",
    "notes": "Expected 500 if API tier/time window provides enough tweets",
})

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(QUALITY_CSV, index=False)

print("Quality report saved:", QUALITY_CSV)
print(quality_df.to_string(index=False))

Quality report saved: q5_tweets_quality_report.csv
                 check_name  value status                                                                                                                                           notes
       required_columns_raw      0   PASS                                                                                                                                   Missing: none
     required_columns_clean      0   PASS                                                                                                                                   Missing: none
              api_preflight      0   WARN Preflight API test status. 402 Payment Required\nYour enrolled account [2028714052411949056] does not have any credits to fulfill this request.
  missing_or_blank_tweet_id      0   PASS                                                                                                                         Raw dataset blank check
  missing_or_blank_

In [ ]:
print("Output Evidence")
print("Raw CSV:", RAW_CSV)
print("Clean CSV:", CLEAN_CSV)
print("Quality CSV:", QUALITY_CSV)

print("\nRaw head:")
if raw_df.empty:
    print("Raw dataset is empty (likely no valid API token/access yet).")
else:
    print(raw_df.head(5).to_string(index=False))

print("\nClean head:")
if clean_df.empty:
    print("Clean dataset is empty.")
else:
    print(clean_df[["tweet_id", "username", "text", "clean_text", "tokens"]].head(5).to_string(index=False))

print("\nQuality report:")
print(quality_df.to_string(index=False))

Output Evidence
Raw CSV: q5_tweets_raw.csv
Clean CSV: q5_tweets_clean.csv
Quality CSV: q5_tweets_quality_report.csv

Raw head:
Raw dataset is empty (likely no valid API token/access yet).

Clean head:
Clean dataset is empty.

Quality report:
                 check_name  value status                                                                                                                                           notes
       required_columns_raw      0   PASS                                                                                                                                   Missing: none
     required_columns_clean      0   PASS                                                                                                                                   Missing: none
              api_preflight      0   WARN Preflight API test status. 402 Payment Required\nYour enrolled account [2028714052411949056] does not have any credits to fulfill this request.
  missing_or_b

In [ ]:
try:
    from google.colab import files
    files.download(RAW_CSV)
    files.download(CLEAN_CSV)
    files.download(QUALITY_CSV)
except Exception:
    print("Skipping download step.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

On the first question, I tried using semantic scholar and had issues with rate limiting. That was something relatively new that I had to learn. I had to try different variations before I stuck with pulling from IMDB. I thought that it was interesting how comparing outputs at each cleaning step and seeing how results can change depending on preprocessing decisions. Handling pagination was tricky and made parts of the assignments tricky, as well as working with APIs in general. Overall, although the assignment was time consuming, I felt like I learned a lot.